## Category important words & similarity search

In [1]:
import pandas as pd
import numpy as np
import re
import time
import nltk
#from nltk import bigrams, trigrams
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity


#model = SentenceTransformer('/Users/zphilipp/git/research/relevance/models/sentence-transformer.model')
model = SentenceTransformer('all-MiniLM-L6-v2')

#pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 200)

prepositions_and_conjunctions = [
    "about", "above", "across", "after", "against", "along", "among", "around", "at",
    "before", "behind", "below", "beneath", "beside", "between", "beyond", "by",
    "during", "for", "from", "in", "inside", "into", "near", "of", "off", "on",
    "out", "outside", "over", "through", "throughout", "to", "toward", "under",
    "until", "up", "with", "within", "without", "and", "but", "or", "for", "nor",
    "so", "yet", "although", "because", "as", "since", "unless", "while", "when",
    "where", "after", "before", "the", "a"
]
pattern = r'\b(?:' + '|'.join(prepositions_and_conjunctions) + r')\b'

def remove_prepositions_and_conjunctions(text):
    text = text.lower()
    cleaned_text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    cleaned_text = re.sub(r'\d+', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    return cleaned_text.replace("-", "")

/Users/zphilipp/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/zphilipp/miniconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [3]:
df = pd.read_csv('category_data.csv')
df.head()

,attributes.v2_category_name,attributes.v1_category_name,parent,guid,description,name,header,category
0,Shopping,Retail,0ed8f46e-2990-448c-9a8c-50665498a84c,7552494f-2a02-4bf8-91b3-ba34d90debdf,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Personal Care,Shopping
1,NaN,NaN,NaN,2b2dbf9e-86d0-4c7e-915b-408c4564df53,Subscriptions,Subscriptions,NaN,NaN
2,Shopping,Retail,71912b5b-4809-4536-bc03-8ee270060b96,36a0ea5b-210a-452f-82ad-e9b6c96c67ab,Home & Garden - Decorative Vases,Home & Garden - Decorative Vases,Home Decor,Shopping
3,Shopping,Retail,8efb9d2c-0947-44d9-af8f-eec79a5a3fc2,2073934d-da10-4cdf-bed1-59bb12e9d00a,Apparel & Accessories - Girls - Belts,Apparel & Accessories - Girls - Belts,Clothing Accessories,Shopping
4,Shopping,Retail,d05bb230-5f44-4180-9f9b-323f72649552,508d94c4-3851-4b51-b0c0-4efcf6381fcb,Action Figure / Doll,Action Figure / Doll,Toys & Hobbies,Shopping


In [5]:
df['text'] = df['description'] + '. ' + df['name']
df['text'] = df['text'].apply(remove_prepositions_and_conjunctions)
df_ = df
df_.head()

,attributes.v2_category_name,attributes.v1_category_name,parent,guid,description,name,header,category,text
0,Shopping,Retail,0ed8f46e-2990-448c-9a8c-50665498a84c,7552494f-2a02-4bf8-91b3-ba34d90debdf,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Personal Care,Shopping,health & beauty deodorant & antiperspirant mens deodorant. health & beauty deodorant & antiperspirant mens deodorant
1,NaN,NaN,NaN,2b2dbf9e-86d0-4c7e-915b-408c4564df53,Subscriptions,Subscriptions,NaN,NaN,subscriptions. subscriptions
2,Shopping,Retail,71912b5b-4809-4536-bc03-8ee270060b96,36a0ea5b-210a-452f-82ad-e9b6c96c67ab,Home & Garden - Decorative Vases,Home & Garden - Decorative Vases,Home Decor,Shopping,home & garden decorative vases. home & garden decorative vases
3,Shopping,Retail,8efb9d2c-0947-44d9-af8f-eec79a5a3fc2,2073934d-da10-4cdf-bed1-59bb12e9d00a,Apparel & Accessories - Girls - Belts,Apparel & Accessories - Girls - Belts,Clothing Accessories,Shopping,apparel & accessories girls belts. apparel & accessories girls belts
4,Shopping,Retail,d05bb230-5f44-4180-9f9b-323f72649552,508d94c4-3851-4b51-b0c0-4efcf6381fcb,Action Figure / Doll,Action Figure / Doll,Toys & Hobbies,Shopping,action figure / doll. action figure / doll


#### Get all titles from Deals and Options text

#### Create word embedings and transform data

In [6]:
df_['text'] = df_['text'].tolist()
df_['embeddings'] = df_['text'].apply(lambda x: model.encode(x))

combined_embeddings = np.array(df_['embeddings'].tolist())

In [9]:
df_.head()
#df_.count()

,attributes.v2_category_name,attributes.v1_category_name,parent,guid,description,name,header,category,text,embeddings
0,Shopping,Retail,0ed8f46e-2990-448c-9a8c-50665498a84c,7552494f-2a02-4bf8-91b3-ba34d90debdf,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Health & Beauty - Deodorant & Antiperspirant - Mens - Deodorant,Personal Care,Shopping,health & beauty deodorant & antiperspirant mens deodorant. health & beauty deodorant & antiperspirant mens deodorant,"[-0.0009589831, 0.0081701465, 0.10928768, 0.015979808, 0.079705745, -0.054364037, 0.08232258, -0.03318259, -0.06707983, -0.013876773, 0.06261128, -0.051048968, -0.01748341, -0.05335682, 0.11192895..."
1,NaN,NaN,NaN,2b2dbf9e-86d0-4c7e-915b-408c4564df53,Subscriptions,Subscriptions,NaN,NaN,subscriptions. subscriptions,"[-0.051509548, -0.06444815, -0.04112121, 0.042434275, -0.0010615113, 0.035495438, 0.08559268, -0.03848302, 0.07432327, 0.0006714391, 0.021693816, 0.03552054, 0.06964188, 0.014078247, -0.004548607,..."
2,Shopping,Retail,71912b5b-4809-4536-bc03-8ee270060b96,36a0ea5b-210a-452f-82ad-e9b6c96c67ab,Home & Garden - Decorative Vases,Home & Garden - Decorative Vases,Home Decor,Shopping,home & garden decorative vases. home & garden decorative vases,"[0.051288802, 0.020537999, 0.06493606, -0.058544688, -0.058173075, 0.027262967, 0.05664413, -0.015885562, -0.040061068, 0.026442869, -0.040510807, -0.0399215, -0.019836022, 0.03982401, 0.06903698,..."
3,Shopping,Retail,8efb9d2c-0947-44d9-af8f-eec79a5a3fc2,2073934d-da10-4cdf-bed1-59bb12e9d00a,Apparel & Accessories - Girls - Belts,Apparel & Accessories - Girls - Belts,Clothing Accessories,Shopping,apparel & accessories girls belts. apparel & accessories girls belts,"[0.011771777, -0.00048552422, 0.0127890995, 0.012789315, -0.027351007, -0.04309254, 0.08880087, -0.010798826, -0.043110605, -0.0079502845, 0.115006246, -0.009183229, 0.10886563, -0.042723708, 0.05..."
4,Shopping,Retail,d05bb230-5f44-4180-9f9b-323f72649552,508d94c4-3851-4b51-b0c0-4efcf6381fcb,Action Figure / Doll,Action Figure / Doll,Toys & Hobbies,Shopping,action figure / doll. action figure / doll,"[-0.013929551, -0.06954958, 0.0118156215, -0.0005743038, -0.02140315, 0.0012134686, 0.06832989, 0.009302696, 0.03917064, 0.058662686, 0.09623468, -0.003958322, 0.0002645173, 0.06230961, 0.06897391..."


In [148]:
def query_embedding_reduce(query_embedding):
    if query_embedding.shape[1] > 384:
        query_embedding_reduced = np.mean(query_embedding.reshape(-1, 2, 384), axis=1)
    else:
        query_embedding_reduced = query_embedding
    return query_embedding_reduced

df_[['id', 'name', 'category', 'text', 'embeddings']].to_csv('models/category_embeding_top.csv')
df_.head(5)g

,Unnamed: 0,id,name,path,text,embeddings
0,0,1bf81df3-efc0-44f3-937f-8d19e12e1530,Nightlife,nearby things to do nightlife,nearby things do nightlife,"[0.10719261, -0.009873317, -0.002886109, 0.04016389, 0.02574348, -0.055390447, 0.04755237, -0.09206853, 0.041121837, -0.024387462, 0.060728706, -0.048695356, -0.0129105635, 0.0455808, 0.107071236,..."
1,1,2afb5ba6-75bc-497e-a53a-be118d00f59a,Sightseeing & Tours,nearby things to do sightseeing & tours,nearby things do sightseeing & tours,"[0.110048324, -0.03330038, 0.05407769, 0.00047145097, 0.025682904, -0.054179642, 0.056862134, -0.032360036, -0.055944394, 0.0068316287, 0.0582175, 0.0161768, -0.010869576, 0.10275831, 0.07755927, ..."
2,2,634fe6f0-7ac5-4ca0-96d5-7f63f97f9cd2,Bus Tours & Rentals,nearby things to do sightseeing & tours bus tours & rentals,nearby things do sightseeing & tours bus tours & rentals,"[0.10644985, -0.04905953, 0.054852676, 0.016697181, -0.035579592, 0.006469342, 0.075778306, -0.025403628, -0.022825733, -0.024200391, 0.04719262, 0.053419624, 0.02457347, 0.09207393, 0.08980535, -..."
3,3,8fce0ab5-5a6a-480d-ac35-f1586b66f857,Brows & Lashes,nearby beauty & spas brows & lashes,nearby beauty & spas brows & lashes,"[0.010597926, -0.052663118, 0.064909294, 0.060403436, -0.049161762, -0.013531411, 0.0118945055, -0.06260367, -0.1079998, -0.028962698, 0.12617949, -0.0657222, -0.046697628, 0.047157846, 0.04524961..."
4,4,91e47ef0-776d-4eaf-9acb-acd360036b94,Eyelash Extensions,nearby beauty & spas brows & lashes eyelash extensions,nearby beauty & spas brows & lashes eyelash extensions,"[-0.010798341, -0.038259745, 0.0706647, 0.06036469, -0.017027609, -0.033844322, 0.0064394977, -0.033618566, -0.09275439, -0.037446145, 0.15190047, -0.03374619, -0.03807696, 0.041630223, 0.06628009..."


In [149]:
def get_top_similarity(query_embedding_reduced, combined_embeddings):
    similarities = cosine_similarity(query_embedding_reduced, combined_embeddings).flatten()
    closest_indices = np.argsort(similarities)[-10:]

    closest_rows = []
    for index in reversed(closest_indices):
        if similarities[index] > 0.3:
        
            closest_rows.append([df_.iloc[index], similarities[index]])

    return closest_rows

### Test query -> category use Cosine similarity of category embedings and query embedings

In [150]:
def get_sim(query):
    start_time = time.time()
    query_embedding_reduced = query_embedding_reduce(model.encode(query).reshape(1, -1))
    print (f"Embeding time :{time.time() - start_time}")
    result = get_top_similarity(query_embedding_reduced, combined_embeddings)
    print (f"Total run time :{time.time() - start_time}")
    for row in result:
        print(f"Closest Category: <{row[0]['name']}> -> score {row[1]}")

In [151]:
get_sim(['massage', 'oil'])

Embeding time :0.23060297966003418
Total run time :0.2340869903564453
Closest Category: <Massage> -> score 0.5936518907546997
Closest Category: <Massage> -> score 0.5578334331512451
Closest Category: <Massage> -> score 0.545110821723938
Closest Category: <Deep Tissue Massage> -> score 0.5422124266624451
Closest Category: <Full Body Massage> -> score 0.5116969347000122
Closest Category: <Custom Massage> -> score 0.5020962953567505
Closest Category: <Reflexology> -> score 0.49334287643432617
Closest Category: <Swedish Massage> -> score 0.48978015780448914
Closest Category: <Hot Stone Massage> -> score 0.47802236676216125
Closest Category: <Couples Massage> -> score 0.4769304394721985


In [152]:
get_sim(['oil'])

Embeding time :0.04117608070373535
Total run time :0.04410696029663086
Closest Category: <Oil Change> -> score 0.42151808738708496


In [153]:
get_sim(['change'])

Embeding time :0.029695987701416016
Total run time :0.03649616241455078


In [154]:
get_sim(['oil', 'change'])

Embeding time :0.016746997833251953
Total run time :0.019485950469970703
Closest Category: <Oil Change> -> score 0.37282800674438477


In [155]:
get_sim(['sauna', 'massage'])

Embeding time :0.1156771183013916
Total run time :0.11664509773254395
Closest Category: <Deep Tissue Massage> -> score 0.6411435008049011
Closest Category: <Full Body Massage> -> score 0.6267561912536621
Closest Category: <Massage> -> score 0.6256269216537476
Closest Category: <Reflexology> -> score 0.6165924072265625
Closest Category: <Custom Massage> -> score 0.5982100963592529
Closest Category: <Massage> -> score 0.5961123704910278
Closest Category: <Saunas> -> score 0.5924926996231079
Closest Category: <Hot Stone Massage> -> score 0.5724921226501465
Closest Category: <Swedish Massage> -> score 0.5599960684776306
Closest Category: <Massage> -> score 0.5589667558670044


In [156]:
get_sim(['oil', 'massage'])

Embeding time :0.028002023696899414
Total run time :0.0371851921081543
Closest Category: <Massage> -> score 0.5936518907546997
Closest Category: <Massage> -> score 0.5578334331512451
Closest Category: <Massage> -> score 0.545110821723938
Closest Category: <Deep Tissue Massage> -> score 0.5422124266624451
Closest Category: <Full Body Massage> -> score 0.5116969347000122
Closest Category: <Custom Massage> -> score 0.5020962953567505
Closest Category: <Reflexology> -> score 0.49334287643432617
Closest Category: <Swedish Massage> -> score 0.48978015780448914
Closest Category: <Hot Stone Massage> -> score 0.47802236676216125
Closest Category: <Couples Massage> -> score 0.4769304394721985


In [157]:
get_sim(['massage', 'oil'])

Embeding time :0.011903047561645508
Total run time :0.01739811897277832
Closest Category: <Massage> -> score 0.5936518907546997
Closest Category: <Massage> -> score 0.5578334331512451
Closest Category: <Massage> -> score 0.545110821723938
Closest Category: <Deep Tissue Massage> -> score 0.5422124266624451
Closest Category: <Full Body Massage> -> score 0.5116969347000122
Closest Category: <Custom Massage> -> score 0.5020962953567505
Closest Category: <Reflexology> -> score 0.49334287643432617
Closest Category: <Swedish Massage> -> score 0.48978015780448914
Closest Category: <Hot Stone Massage> -> score 0.47802236676216125
Closest Category: <Couples Massage> -> score 0.4769304394721985


In [158]:
get_sim(['valvoline', 'oil'])

Embeding time :0.028905868530273438
Total run time :0.029619932174682617
Closest Category: <Oil Change> -> score 0.3273688554763794
Closest Category: <Massage> -> score 0.3056454360485077


In [159]:
get_sim(['water'])

Embeding time :0.014217138290405273
Total run time :0.017487049102783203
Closest Category: <Water Sports> -> score 0.36752113699913025
Closest Category: <Colonic Hydrotherapy> -> score 0.31639203429222107


In [160]:
get_sim(['water', 'parks'])

Embeding time :0.021202802658081055
Total run time :0.05355095863342285
Closest Category: <Water Sports> -> score 0.4425596594810486
Closest Category: <Sports & Outdoors> -> score 0.3915403485298157
Closest Category: <Sports & Outdoors> -> score 0.35566163063049316
Closest Category: <Fun & Leisure> -> score 0.3468207120895386
Closest Category: <Yoga> -> score 0.33923429250717163
Closest Category: <Golf> -> score 0.31867673993110657
Closest Category: <Amusement Parks> -> score 0.31864893436431885
Closest Category: <Escape Games> -> score 0.31035852432250977
Closest Category: <Wine> -> score 0.3075743317604065
Closest Category: <Golf> -> score 0.30693745613098145


In [161]:
get_sim(['amc'])

Embeding time :0.018496036529541016
Total run time :0.022899866104125977


In [162]:
get_sim(['pilates'])

Embeding time :0.02478623390197754
Total run time :0.037889957427978516
Closest Category: <Pilates> -> score 0.6689449548721313
Closest Category: <Plastic Surgery> -> score 0.4212318956851959
Closest Category: <Yoga> -> score 0.368234783411026
Closest Category: <Chiropractor> -> score 0.354994535446167
Closest Category: <Medical> -> score 0.352630615234375
Closest Category: <Acupuncture> -> score 0.31720924377441406


In [163]:
get_sim(['ring'])

Embeding time :0.011501073837280273
Total run time :0.02244400978088379


In [164]:
get_sim(['wheel'])

Embeding time :0.028411865234375
Total run time :0.02939915657043457
Closest Category: <Tires & Wheels> -> score 0.44538387656211853


In [165]:
get_sim(['nail'])

Embeding time :0.011105060577392578
Total run time :0.014129161834716797
Closest Category: <Mani Pedi> -> score 0.376738965511322


In [166]:
get_sim(['massage', 'palace'])

Embeding time :0.015129804611206055
Total run time :0.021019935607910156
Closest Category: <Massage> -> score 0.6104089021682739
Closest Category: <Full Body Massage> -> score 0.6042941808700562
Closest Category: <Deep Tissue Massage> -> score 0.5990667343139648
Closest Category: <Custom Massage> -> score 0.5824674367904663
Closest Category: <Reflexology> -> score 0.5727773904800415
Closest Category: <Hot Stone Massage> -> score 0.5465376377105713
Closest Category: <Massage> -> score 0.542343258857727
Closest Category: <Couples Massage> -> score 0.5377268195152283
Closest Category: <Swedish Massage> -> score 0.5308575630187988
Closest Category: <Massage> -> score 0.5209032893180847


In [167]:
get_sim(['apple', 'cider'])

Embeding time :0.032708168029785156
Total run time :0.04047203063964844
Closest Category: <Electronics> -> score 0.41071468591690063
Closest Category: <Electronics> -> score 0.3510616719722748
Closest Category: <Wine> -> score 0.34688231348991394
Closest Category: <Retail> -> score 0.3149658441543579
